
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

In [0]:
%run ../Includes/Copy-Datasets


## Exploring The Source dDirectory

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")
display(files)

path,name,size,modificationTime
dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet,01.parquet,18823,1787177710000
dbfs:/mnt/demo-datasets/bookstore/orders-raw/02.parquet,02.parquet,18814,1787196156000
dbfs:/mnt/demo-datasets/bookstore/orders-raw/03.parquet,03.parquet,18822,1787196200000



## Auto Loader

In [0]:
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "dbfs:/mnt/demo/checkpoints/orders_raw")
    .load(f"{dataset_bookstore}/orders-raw")
    .createOrReplaceTempView("orders_raw_temp"))


## Enriching Raw Data

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_tmp AS (
  SELECT *, current_timestamp() arrival_time, input_file_name() source_file
  FROM orders_raw_temp
)

In [0]:
%sql
SELECT * FROM orders_tmp

order_id,order_timestamp,customer_id,quantity,total,books,_rescued_data,arrival_time,source_file
000000000006341,1657520256,C00788,1,41,"List(List(B08, 1, 41))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006342,1657520256,C00788,1,41,"List(List(B08, 1, 41))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006343,1657531717,C00654,1,28,"List(List(B02, 1, 28))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006344,1657531717,C00654,1,28,"List(List(B02, 1, 28))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006345,1657543676,C00762,1,49,"List(List(B01, 1, 49))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006346,1657543676,C00762,1,49,"List(List(B01, 1, 49))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006347,1657546079,C01014,1,28,"List(List(B02, 1, 28))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006348,1657546658,C00633,1,24,"List(List(B09, 1, 24))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006349,1657546658,C00633,1,24,"List(List(B09, 1, 24))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet
000000000006350,1657547177,C00638,1,35,"List(List(B03, 1, 35))",null,2026-08-20T03:38:56.781Z,dbfs:/mnt/demo-datasets/bookstore/orders-raw/01.parquet


## Creating Bronze Table

In [0]:
(spark.table("orders_tmp")
      .writeStream
      .format("delta")
      .option("checkpointLocation", "dbfs:/mnt/demo/checkpoints/orders_bronze")
      .outputMode("append")
      .table("orders_bronze"))

In [0]:
%sql
SELECT count(*) FROM orders_bronze

count(1)
4000


In [0]:
load_new_data()

Loading 04.parquet file to the bookstore dataset



#### Creating Static Lookup Table

In [0]:
(spark.read
      .format("json")
      .load(f"{dataset_bookstore}/customers-json")
      .createOrReplaceTempView("customers_lookup"))

In [0]:
%sql
SELECT * FROM customers_lookup

customer_id,email,profile,updated
C00301,thomas.lane@gmail.com,"{""first_name"":""Thomas"",""last_name"":""Lane"",""gender"":""Male"",""address"":{""street"":""06 Boulevard Victor Hugo"",""city"":""Paris"",""country"":""France""}}",2021-12-14T23:15:43.375Z
C00302,ocolegatele@blogger.com,"{""first_name"":""Odilia"",""last_name"":""Colegate"",""gender"":""Female"",""address"":{""street"":""07 Sommers Parkway"",""city"":""Lyon"",""country"":""France""}}",2021-12-14T23:15:43.375Z
C00303,acolledged2@nbcnews.com,"{""first_name"":""Andros"",""last_name"":""Colledge"",""gender"":""Male"",""address"":{""street"":""342 Katie Center"",""city"":""Gort"",""country"":""Ireland""}}",2021-12-14T23:15:43.375Z
C00304,null,"{""first_name"":""Iver"",""last_name"":""Collet"",""gender"":""Male"",""address"":{""street"":""12126 Union Point"",""city"":""Iguape"",""country"":""Brazil""}}",2021-12-14T23:15:43.375Z
C00305,pcollier5r@cmu.edu,"{""first_name"":""Page"",""last_name"":""Collier"",""gender"":""Male"",""address"":{""street"":""3 Farragut Lane"",""city"":""Berlin"",""country"":""Germany""}}",2021-12-14T23:15:43.375Z
C00306,null,"{""first_name"":""Tally"",""last_name"":""Collins"",""gender"":""Male"",""address"":{""street"":""4 Hovde Park"",""city"":""Cairo"",""country"":""Egypt""}}",2021-12-14T23:15:43.375Z
C00307,lcollocottcm@t-online.de,"{""first_name"":""Leupold"",""last_name"":""Collocott"",""gender"":""Male"",""address"":{""street"":""917 Stephen Circle"",""city"":""Dzerzhinskiy"",""country"":""Russia""}}",2021-12-14T23:15:43.375Z
C00308,icolloughfa@prweb.com,"{""first_name"":""Inesita"",""last_name"":""Collough"",""gender"":""Female"",""address"":{""street"":""7910 Delladonna Street"",""city"":""Osoyoos"",""country"":""Canada""}}",2021-12-14T23:15:43.375Z
C00309,jcollymore4n@pcworld.com,"{""first_name"":""Joelle"",""last_name"":""Collymore"",""gender"":""Female"",""address"":{""street"":""19 Dayton Court"",""city"":""Yidu"",""country"":""China""}}",2021-12-14T23:15:43.375Z
C00310,gcolnetef@japanpost.jp,"{""first_name"":""Goldi"",""last_name"":""Colnet"",""gender"":""Female"",""address"":{""street"":""710 Knutson Place"",""city"":""Suso"",""country"":""Philippines""}}",2021-12-14T23:15:43.375Z


## Creating Silver Table

In [0]:
(spark.readStream
  .table("orders_bronze")
  .createOrReplaceTempView("orders_bronze_tmp"))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_enriched_tmp AS (
  SELECT order_id, quantity, o.customer_id, c.profile:first_name as f_name, c.profile:last_name as l_name,
         cast(from_unixtime(order_timestamp, 'yyyy-MM-dd HH:mm:ss') AS timestamp) order_timestamp, books
  FROM orders_bronze_tmp o
  INNER JOIN customers_lookup c
  ON o.customer_id = c.customer_id
  WHERE quantity > 0)

In [0]:
(spark.table("orders_enriched_tmp")
      .writeStream
      .format("delta")
      .option("checkpointLocation", "dbfs:/mnt/demo/checkpoints/orders_silver")
      .outputMode("append")
      .table("orders_silver"))

In [0]:
%sql
SELECT * FROM orders_silver

order_id,quantity,customer_id,f_name,l_name,order_timestamp,books
000000000009397,1,C00494,Sherlocke,Fairbard,2022-07-12T17:13:57Z,"List(List(B08, 1, 41))"
000000000009406,1,C00495,Kerrie,Falcus,2022-07-12T19:17:02Z,"List(List(B09, 1, 24))"
000000000009984,1,C00496,Dacie,Fallens,2022-07-23T17:16:31Z,"List(List(B11, 1, 38))"
000000000009834,1,C00497,Waldon,Falshaw,2022-07-21T13:59:44Z,"List(List(B09, 1, 24))"
000000000009755,1,C00498,Gussi,Fancy,2022-07-20T07:03:40Z,"List(List(B09, 1, 24))"
000000000009785,1,C00499,Wenona,Farguhar,2022-07-20T19:31:00Z,"List(List(B02, 1, 28))"
000000000009552,1,C00500,Colman,Farncomb,2022-07-16T12:05:11Z,"List(List(B09, 1, 24))"
000000000010299,1,C00501,Wilfrid,Fedoronko,2022-07-28T14:52:28Z,"List(List(B09, 1, 24))"
000000000010166,1,C00502,Elke,Feedome,2022-07-26T01:48:17Z,"List(List(B08, 1, 41))"
000000000009441,1,C00503,Felicdad,Feifer,2022-07-13T19:11:44Z,"List(List(B07, 1, 33))"


In [0]:
%sql
SELECT COUNT(*) FROM orders_silver

count(1)
5000


In [0]:
load_new_data()

Loading 05.parquet file to the bookstore dataset


## Creating Gold Table

In [0]:
(spark.readStream
  .table("orders_silver")
  .createOrReplaceTempView("orders_silver_tmp"))

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_customer_books_tmp AS (
  SELECT customer_id, f_name, l_name, date_trunc("DD", order_timestamp) order_date, sum(quantity) books_counts
  FROM orders_silver_tmp
  GROUP BY customer_id, f_name, l_name, date_trunc("DD", order_timestamp)
  )

In [0]:
(spark.table("daily_customer_books_tmp")
      .writeStream
      .format("delta")
      .outputMode("complete")
      .option("checkpointLocation", "dbfs:/mnt/demo/checkpoints/daily_customer_books")
      .trigger(availableNow=True)
      .table("daily_customer_books"))

In [0]:
%sql
SELECT * FROM daily_customer_books

customer_id,f_name,l_name,order_date,books_counts
C01162,Bran,Oldall,2022-07-26T00:00:00Z,6
C00906,Bea,Libbie,2022-07-23T00:00:00Z,6
C00889,Dyana,Ledingham,2022-07-23T00:00:00Z,6
C00685,Loralyn,Heater,2022-07-13T00:00:00Z,12
C00711,Jerrold,Huggon,2022-07-20T00:00:00Z,12
C00567,Hieronymus,Gavrielly,2022-07-30T00:00:00Z,12
C00690,Graig,Heister,2022-07-24T00:00:00Z,12
C00721,Rhoda,Iacofo,2022-07-27T00:00:00Z,12
C00793,Kendall,Kemmer,2022-07-23T00:00:00Z,12
C00934,Evie,Lowdeane,2022-07-25T00:00:00Z,6


In [0]:
load_new_data()

Loading 06.parquet file to the bookstore dataset



## Stopping active streams

In [0]:
for s in spark.streams.active:
    print("Stopping stream: " + s.id)
    s.stop()
    s.awaitTermination()

Stopping stream: e4496a0c-9fe4-4534-ae80-cef1a972d1b5
Stopping stream: d99ea1da-45a7-463d-ad67-7f7315b21b92
